### Step 1: Kaggle Setup and Dataset Download
Upload your `kaggle.json` file when prompted. This will configure the Kaggle API and download the NIH dataset.

### Step 2: Create Binary Labels and Patient-Level Splits
This script reads the metadata, creates a binary label (0 for Normal, 1 for Abnormal), and splits the dataset by `Patient ID` so that no patient overlaps between the training and validation sets.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import glob
import os

def create_splits(data_dir):
    csv_path = os.path.join(data_dir, 'Data_Entry_2017.csv')
    if not os.path.exists(csv_path):
        print(f"CSV not found at {csv_path}. Make sure the dataset is extracted.")
        return

    # Load metadata
    df = pd.read_csv(csv_path)

    # Create binary labels: 'No Finding' -> 0 (Normal), everything else -> 1 (Abnormal)
    df['label'] = df['Finding Labels'].apply(lambda x: 0.0 if x == 'No Finding' else 1.0)

    # Get all image paths
    # The NIH dataset unzips images into multiple folders (images_001, images_002, etc.)
    image_paths = glob.glob(os.path.join(data_dir, 'images_*', 'images', '*.png'))
    path_dict = {os.path.basename(p): p for p in image_paths}

    # Map full image paths to the dataframe
    df['image_path'] = df['Image Index'].map(path_dict)

    # Drop rows where we couldn't find the image (just in case)
    df = df.dropna(subset=['image_path'])

    # --- Patient-Level Split ---
    # Get unique patient IDs
    unique_patients = df['Patient ID'].unique()

    # Split patients into 80% train, 20% val
    train_patients, val_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

    # Filter dataframe based on patient splits
    train_df = df[df['Patient ID'].isin(train_patients)][['image_path', 'label']]
    val_df = df[df['Patient ID'].isin(val_patients)][['image_path', 'label']]

    # Save to CSV
    train_df.to_csv('nih_train.csv', index=False)
    val_df.to_csv('nih_val.csv', index=False)

    print(f"Created nih_train.csv with {len(train_df)} images.")
    print(f"Created nih_val.csv with {len(val_df)} images.")
    display(train_df.head())

# Call the function using the dynamic path from kagglehub
create_splits(dataset_path)

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from PIL import Image
from google.colab import files
files.upload()


class ChestXrayDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        # Read the CSV that contains image_path and label columns
        self.dataframe = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        # Number of rows in the CSV
        return len(self.dataframe)

    def __getitem__(self, index):
        # Read one row
        row = self.dataframe.iloc[index]

        # Load the image and force it into 3 channels
        image = Image.open(row["image_path"]).convert("RGB")

        # Read the label as float for BCEWithLogitsLoss
        label = float(row["label"])

        # Apply transforms if provided
        if self.transform is not None:
            image = self.transform(image)

        # Return tensor image and tensor label
        label_tensor = torch.tensor(label, dtype=torch.float32)
        return image, label_tensor

In [ ]:
from torchvision.models import densenet121
from torchvision.models import DenseNet121_Weights


def build_binary_model(freeze_backbone=False):
    # Load ImageNet-pretrained DenseNet-121
    model = densenet121(weights=DenseNet121_Weights.DEFAULT)

    # DenseNet stores the last classification layer here
    in_features = model.classifier.in_features

    # Replace the 1000-class head with a 1-output binary head
    model.classifier = nn.Linear(in_features, 1)

    # Optional: freeze the backbone first for easier debugging
    if freeze_backbone:
        for parameter in model.features.parameters():
            parameter.requires_grad = False

    return model

In [ ]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        logits = model(images).squeeze(1)

        # Compute binary classification loss
        loss = loss_fn(logits, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    average_loss = total_loss / total_examples
    return average_loss

In [ ]:
import torch
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()

    all_labels = []
    all_probs = []

    for images, labels in dataloader:
        images = images.to(device)

        logits = model(images).squeeze(1)
        probabilities = torch.sigmoid(logits).cpu()

        labels_cpu = labels.cpu()

        index = 0
        while index < len(labels_cpu):
            all_labels.append(float(labels_cpu[index].item()))
            all_probs.append(float(probabilities[index].item()))
            index += 1

    # Convert probabilities into 0/1 predictions at threshold 0.5
    all_predictions = []
    index = 0
    while index < len(all_probs):
        if all_probs[index] >= 0.5:
            all_predictions.append(1)
        else:
            all_predictions.append(0)
        index += 1

    accuracy = accuracy_score(all_labels, all_predictions)
    auroc = roc_auc_score(all_labels, all_probs)

    return {
        "accuracy": accuracy,
        "auroc": auroc,
    }

In [ ]:
import torch
import torch.nn as nn


def collect_batch_norm_parameters(model):
    parameters = []

    # Freeze everything first
    for parameter in model.parameters():
        parameter.requires_grad = False

    # Only enable BatchNorm scale and shift
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            if module.weight is not None:
                module.weight.requires_grad = True
                parameters.append(module.weight)

            if module.bias is not None:
                module.bias.requires_grad = True
                parameters.append(module.bias)

            # Use current batch statistics during adaptation
            module.track_running_stats = False
            module.running_mean = None
            module.running_var = None

    return parameters


def binary_entropy_from_logits(logits):
    # logits shape: [batch]
    probabilities = torch.sigmoid(logits)

    epsilon = 1e-6

    # Bernoulli entropy:
    # -p log p - (1-p) log (1-p)
    entropy = -(
        probabilities * torch.log(probabilities + epsilon)
        + (1.0 - probabilities) * torch.log(1.0 - probabilities + epsilon)
    )

    return entropy

In [ ]:
import torch


class EntropyGatedAdapter:
    def __init__(self, model, optimizer, entropy_threshold):
        self.model = model
        self.optimizer = optimizer
        self.entropy_threshold = entropy_threshold

    @torch.enable_grad()
    def forward_and_adapt(self, images):
        # Test-time adaptation needs train mode for BatchNorm updates
        self.model.train()

        # Forward pass
        logits = self.model(images).squeeze(1)

        # Compute per-sample prediction entropy
        entropies = binary_entropy_from_logits(logits)

        # Only adapt on confident samples
        selected_mask = entropies < self.entropy_threshold

        selected_count = int(selected_mask.sum().item())

        if selected_count > 0:
            selected_entropies = entropies[selected_mask]

            # Minimize entropy only on selected samples
            loss = selected_entropies.mean()

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        return logits.detach(), entropies.detach(), selected_mask.detach()

In [ ]:
import torch
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


def evaluate_with_adapter(adapter, dataloader, device):
    all_labels = []
    all_probs = []

    total_seen = 0
    total_selected = 0

    for images, labels in dataloader:
        images = images.to(device)

        logits, entropies, selected_mask = adapter.forward_and_adapt(images)

        probabilities = torch.sigmoid(logits).cpu()
        labels_cpu = labels.cpu()
        selected_cpu = selected_mask.cpu()

        batch_index = 0
        while batch_index < len(labels_cpu):
            all_labels.append(float(labels_cpu[batch_index].item()))
            all_probs.append(float(probabilities[batch_index].item()))

            total_seen += 1
            if bool(selected_cpu[batch_index].item()):
                total_selected += 1

            batch_index += 1

    all_predictions = []
    index = 0
    while index < len(all_probs):
        if all_probs[index] >= 0.5:
            all_predictions.append(1)
        else:
            all_predictions.append(0)
        index += 1

    accuracy = accuracy_score(all_labels, all_predictions)
    auroc = roc_auc_score(all_labels, all_probs)

    selection_rate = total_selected / total_seen

    return {
        "accuracy": accuracy,
        "auroc": auroc,
        "selection_rate": selection_rate,
    }

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    source_train_dataset = ChestXrayDataset("source_train.csv", transform=train_transform)
    source_val_dataset = ChestXrayDataset("source_val.csv", transform=eval_transform)
    target_test_dataset = ChestXrayDataset("target_test.csv", transform=eval_transform)

    source_train_loader = DataLoader(source_train_dataset, batch_size=16, shuffle=True)
    source_val_loader = DataLoader(source_val_dataset, batch_size=16, shuffle=False)
    target_test_loader = DataLoader(target_test_dataset, batch_size=16, shuffle=False)

    model = build_binary_model(freeze_backbone=False)
    model = model.to(device)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    epoch = 0
    num_epochs = 5

    while epoch < num_epochs:
        train_loss = train_one_epoch(model, source_train_loader, optimizer, loss_fn, device)
        source_metrics = evaluate_model(model, source_val_loader, device)

        print("epoch:", epoch + 1)
        print("train_loss:", train_loss)
        print("source_val_accuracy:", source_metrics["accuracy"])
        print("source_val_auroc:", source_metrics["auroc"])
        print()

        epoch += 1

    # Evaluate target with no adaptation
    target_metrics = evaluate_model(model, target_test_loader, device)
    print("target_no_adaptation:", target_metrics)

    # Build a fresh copy for TTA
    tta_model = build_binary_model(freeze_backbone=False)
    tta_model.load_state_dict(model.state_dict())
    tta_model = tta_model.to(device)

    tta_parameters = collect_batch_norm_parameters(tta_model)
    tta_optimizer = torch.optim.Adam(tta_parameters, lr=1e-4)

    adapter = EntropyGatedAdapter(
        model=tta_model,
        optimizer=tta_optimizer,
        entropy_threshold=0.15,
    )

    adapted_metrics = evaluate_with_adapter(adapter, target_test_loader, device)
    print("target_entropy_gated_tta:", adapted_metrics)


if __name__ == "__main__":
    main()

NameError: name 'ChestXrayDataset' is not defined